Lakeflow Connect - 
- Used for data ingestion 
- earlier organizations have to use a lot of third party tools and inhouse solutions for data ingestions.
We can ingest data from many resources -
- cloud storages - adls gen2,amazon s3,databricks volumes
- datbases
- Saas Applications
With lakeflow connect we can have efficient ingestion pipelines all within databricks

> Cloud Storage / Saas Application ingest into Delta Lake then we perform ETL on data over the Medallian Architecture 
- Bronze -> Silver -> Gold
  - Each layer has different data quality levels.
  - As we go from left to right the data quality improves.
> Lakeflow Connect
- different types of connector -
  - upload files from local storage to volume
    - standard connectors -- ingest from cloud storage into your delta lake
  - managed connectors -- ingest from SaaS applications/Databases.

> STANDARD CONNECTORS -
- different kinds of ingestion modes 
  - batch ingestion ( all files will be added & all data is re ingested at all times.)
  - incremental batch ( only pick the new files to be ingested previously ingested records are ignored.)
  - incremental streaming ( continously load data rows or batches of data row as it is generated so that yiu can query as it arrives in real time)


> [DATA INGESTION FROM CLOUD STORAGE]
[ STANDARD CONNECTORS]
- Data ingestionb from cloud storage 
  - CTAS
  - Copy Into 
  - Auto Loader


> Lakeflow Connect Standard Connectors { Convert raw file into delta tables}

> CTAS --- Create table as select *
- it will not do incremental load it will reload all the data & ingest it by creating it again

In [0]:
%sql
SHOW CATALOGS;
SELECT current_catalog();

current_catalog()
workspace


> Lets create a delta table using CTAS

> We will create the volume 


In [0]:
%sql
Create Volume if not exists CTAS;

In [0]:
%fs
mkdirs /Volumes/workspace/default/CTAS/orders

True

> Now lets add the data into the volume using UI from path E:\Databricks Labs\4515050-Week5-Datasets-2

- Then lets view the files 

In [0]:
%fs
ls /Volumes/workspace/default/ctas/orders

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/ctas/orders/orders1.csv,orders1.csv,224,1766164825000


> WE CAN VIEW THE ORDERS file created

> read_files() --- is the function that creates CTAS

In [0]:
%sql
DROP TABLE IF EXISTS orders;
CREATE TABLE orders AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/ctas/orders',
  format => 'csv'
)

num_affected_rows,num_inserted_rows


Lets view the data now

In [0]:
%sql
SELECT * from orders;

order_id,order_date,customer_id,order_status,_rescued_data
1111111,2013-07-25T00:00:00.000Z,11599,CLOSED,null
1111111,2013-07-25T00:00:00.000Z,256,PENDING_PAYMENT,null
3333333,2013-07-25T00:00:00.000Z,12111,COMPLETE,null
4444444,2013-07-25T00:00:00.000Z,8827,CLOSED,null


LETS VIEW THE HISTORY

In [0]:
%sql
DESCRIBE HISTORY orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2025-12-19T17:25:58.000Z,72777155043042,rishabh.jhingran@outlook.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2605982891571557),1219-165828-3ux55hfa-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 1593)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13


- While using read_files if there is any data that doesnot follow the schema is added to the _rescued_data collumn
  - we will never miss any data.

Now lets upload the second file from the same location of local path

> After uploading now lets create the table again and check the select query results we should ideally see 8 rows 4 from 1st file and 4 from second

In [0]:
%sql
DROP TABLE IF EXISTS orders;
CREATE TABLE orders AS
SELECT * from read_files(
  '/Volumes/workspace/default/ctas/orders/',
  format => 'csv'
);
SELECT * from orders;

order_id,order_date,customer_id,order_status,_rescued_data
1111111,2013-07-25T00:00:00.000Z,11599,CLOSED,null
1111111,2013-07-25T00:00:00.000Z,256,PENDING_PAYMENT,null
3333333,2013-07-25T00:00:00.000Z,12111,COMPLETE,null
4444444,2013-07-25T00:00:00.000Z,8827,CLOSED,null
888888,2013-07-25T00:00:00.000Z,11599,CLOSED,null
999999,2013-07-25T00:00:00.000Z,256,PENDING_PAYMENT,null
3,2013-07-25T00:00:00.000Z,12111,CLOSED,null
4,2013-07-25T00:00:00.000Z,8827,PENDING_PAYMENT,null


> We can see the correct records 

> Now we are going to add another file with an extra column 
- It will perform batch processing for all the 3 csv files everytime and drop the old table & create a new one.
- Now even if there is an extra column it will add it.
- Now it will create a schema for combined data that is the schema aftwer reading all the csv files in the path & add a null wherever it doesnt find a value.

> Upload manually the third file

--- Lets add another column along with dat read from the files _metadata.file_name

In [0]:
%sql
DROP TABLE IF EXISTS orders;
---- use `` to give column alias
CREATE TABLE orders AS
SELECT *,_metadata.file_name `file_name` from read_files(
  '/Volumes/workspace/default/ctas/orders/',
  format => 'csv'
);
SELECT * from orders;

order_id,order_date,customer_id,order_status,order_amount,_rescued_data,file_name
1111111,2013-07-25T00:00:00.000Z,11599,CLOSED,null,null,orders1.csv
1111111,2013-07-25T00:00:00.000Z,256,PENDING_PAYMENT,null,null,orders1.csv
3333333,2013-07-25T00:00:00.000Z,12111,COMPLETE,null,null,orders1.csv
4444444,2013-07-25T00:00:00.000Z,8827,CLOSED,null,null,orders1.csv
888888,2013-07-25T00:00:00.000Z,11599,CLOSED,null,null,orders2.csv
999999,2013-07-25T00:00:00.000Z,256,PENDING_PAYMENT,null,null,orders2.csv
3,2013-07-25T00:00:00.000Z,12111,CLOSED,null,null,orders2.csv
4,2013-07-25T00:00:00.000Z,8827,PENDING_PAYMENT,null,null,orders2.csv
100000,2013-07-25T00:00:00.000Z,11599,CLOSED,10,null,orders3.csv
200000,2013-07-25T00:00:00.000Z,256,PENDING_PAYMENT,20,null,orders3.csv


> WE CAN CLEARLY SEE THAT THERE IS NULL FOR ORDER_AMOUNT FOR THE FIRST TWO FILES BUT VALUES FOR THE LATEST FILE WE ADDED.
- WE CAN FIND OUR COLUMN IF WE SCROLL TO THE RIGHT FILE_NAME

> Now lets us add customers file with csv with a wrong schema 
- Upload orders 4 and perform the same 

In [0]:
%sql
DROP TABLE IF EXISTS orders;
CREATE TABLE orders AS
SELECT *,_metadata.file_name file_name from read_files(
  '/Volumes/workspace/default/ctas/orders/',
  format => 'csv'
);
SELECT * FROM orders;

order_id,order_date,customer_id,order_status,order_amount,_rescued_data,file_name
1111111,2013-07-25T00:00:00.000Z,11599,CLOSED,null,null,orders1.csv
1111111,2013-07-25T00:00:00.000Z,256,PENDING_PAYMENT,null,null,orders1.csv
3333333,2013-07-25T00:00:00.000Z,12111,COMPLETE,null,null,orders1.csv
4444444,2013-07-25T00:00:00.000Z,8827,CLOSED,null,null,orders1.csv
888888,2013-07-25T00:00:00.000Z,11599,CLOSED,null,null,orders2.csv
999999,2013-07-25T00:00:00.000Z,256,PENDING_PAYMENT,null,null,orders2.csv
3,2013-07-25T00:00:00.000Z,12111,CLOSED,null,null,orders2.csv
4,2013-07-25T00:00:00.000Z,8827,PENDING_PAYMENT,null,null,orders2.csv
100000,2013-07-25T00:00:00.000Z,11599,CLOSED,10,null,orders3.csv
200000,2013-07-25T00:00:00.000Z,256,PENDING_PAYMENT,20,null,orders3.csv


> Now if we lok closely customer id was suppose to be int but even after entering wrong schema we arfe getting the data because it converted it into string type

> NOW LETS PROVIDE THE SCHEMA TO THE TABLE 

In [0]:
%sql
drop table if exists orders;
create table orders as
select *,_metadata.file_name from read_files('/Volumes/workspace/default/ctas/orders/',
 format => 'csv',
 schema => 'order_id int, order_date string,customer_id bigint,order_status string
 ,corrupt_data string',
 header => 'true',
 mode => 'PERMISSIVE',
 columnNameofCorruptRecord => 'corrupt_data');
SELECT * from orders;


order_id,order_date,customer_id,order_status,corrupt_data,file_name
1111111,2013-07-25 00:00:00.0,11599,CLOSED,null,orders1.csv
1111111,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,null,orders1.csv
3333333,2013-07-25 00:00:00.0,12111,COMPLETE,null,orders1.csv
4444444,2013-07-25 00:00:00.0,8827,CLOSED,null,orders1.csv
888888,2013-07-25 00:00:00.0,11599,CLOSED,null,orders2.csv
999999,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,null,orders2.csv
3,2013-07-25 00:00:00.0,12111,CLOSED,null,orders2.csv
4,2013-07-25 00:00:00.0,8827,PENDING_PAYMENT,null,orders2.csv
100000,2013-07-25 00:00:00.0,11599,CLOSED,"100000,2013-07-25 00:00:00.0,11599,CLOSED,10",orders3.csv
200000,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,"200000,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,20",orders3.csv


> We will find that if there is any record that is not according to our schema it will convert & put it into corrupt_record column in table.
  - the whole row is kept 


> MODE in read_files
  Mode defines how spark should behave if it founds bad or malformed records.
THREE DIFFERENT TYPES OF MODE -
1. PERMISSIVE - keeps the job running & puts the bad row into _rescued_data.
2. DROP MALFORMED - silently drops bad rows.
3. FAILFAST - aborts on first bad row.


> DRAWBACK -

- it doesnot support incremental batch load.